In [ ]:
# ==========================================
# 13. Week 11: Lengthscale Validation Check
# ==========================================

# CHANGED: Continuing down our precision-drilling trajectory. To prevent the surrogate 
# model from over-clustering inputs and causing numerical errors, we introduced a 
# diagnostic print code check to output the optimized kernel variance theta scores directly.

print("--- Executing Week 11 Code Pipeline for All 8 Functions ---")

week_11_submissions = {
    1: {"X": [0.684361, 0.712365], "Y": 1.0766970877108948e-7},
    2: {"X": [0.710781, 0.931563], "Y": 0.6547427685178244},
    3: {"X": [0.519727, 0.639727, 0.359727], "Y": -0.023719127001304456},
    4: {"X": [0.580332, 0.430332, 0.430332, 0.280332], "Y": -3.820935027747399},
    5: {"X": [0.229991, 0.859991, 0.889991, 0.889991], "Y": 1258.5793193416432},
    6: {"X": [0.747383, 0.177383, 0.747383, 0.717383, 0.047383], "Y": -0.6658542923649372},
    7: {"X": [0.057896, 0.491675, 0.247420, 0.218116, 0.420425, 0.730973], "Y": 1.3649646475908397},
    8: {"X": [0.056441, 0.065961, 0.022925, 0.038788, 0.403931, 0.801061, 0.488306, 0.893087], "Y": 9.5984756473676}
}

for i in range(1, 9):
    X_init = np.load(f'{DATA_PATH}function_{i}_initial_inputs.npy')
    y_init = np.load(f'{DATA_PATH}function_{i}_initial_outputs.npy')
    
    past_x = np.array([hist_X[i][w] for w in range(10)])
    past_y = np.array([hist_Y[i][w] for w in range(10)])
    
    w11_x = np.array([week_11_submissions[i]["X"]])
    w11_y = np.array([week_11_submissions[i]["Y"]])
    
    X_combined = np.vstack((X_init, past_x, w11_x))
    y_combined = np.concatenate((y_init, past_y, w11_y))
    
    hist_X[i].append(week_11_submissions[i]["X"])
    hist_Y[i].append(week_11_submissions[i]["Y"])
    
    # CHANGED: Direct initialization block to read and print internal GPR hyperparameters
    gpr_diag = GaussianProcessRegressor(kernel=Matern(nu=2.5), alpha=1e-6, normalize_y=True, random_state=99)
    gpr_diag.fit(X_combined, y_combined)
    
    bounds = np.array([[0.0, 1.0]] * fn_dimensions[i])
    next_x = run_optimization_pipeline(X_combined, y_combined, bounds, n_restarts=50, xi=0.01)
    
    print(f"Function {i} | Max Y: {np.max(y_combined):.4f} | Diagnostic Log-Theta: {np.round(gpr_diag.kernel_.theta, 2)}")
    print(f" -> Next Recommended Coordinates: {np.round(next_x, 5)}\n")

print("Week 11 diagnostic loops completed successfully.")
